# M3: Keypoints sobre Llantas y Estimación de Número de Ejes (`YOLO11-pose` + `YOLOE`)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Implementa el flujo oficial de 3 etapas del notebook `keypoints_llantas_yolo_detectron2_v3.ipynb`:
1. **Zero-shot (`YOLOE-seg`)**: Prompts `wheel`, `tire`, `car wheel` $\rightarrow$ máscara de cada llanta $\rightarrow$ **5 keypoints geométricos (`cv2.fitEllipse`)**.
2. **Auto-etiquetado y Roboflow**: Genera el dataset en formato `YOLO-pose` con `kpt_shape: [5, 3]` y `flip_idx: [0, 1, 2, 4, 3]` (corregible y versionable en **Roboflow**).
3. **Entrenamiento de `YOLO11s-pose` y Estimación de Ejes**: Entrena el modelo de 5 keypoints por llanta y aplica el algoritmo de asociación llanta $\rightarrow$ vehículo (`UMBRAL_CONTENCION=0.7`, `UMBRAL_CERCANA=0.65`, `SEPARACION_EJE=0.6`) para contar el **número de ejes**.

| # | Nombre del Keypoint | Significado |
|---|---|---|
| `0` | `centro` | Centro del aro |
| `1` | `contacto` | Punto de contacto con el suelo |
| `2` | `superior` | Punto más alto de la llanta |
| `3` | `izquierda` | Extremo izquierdo |
| `4` | `derecha` | Extremo derecho |

In [ ]:
# 1. Instalación de Ultralytics, CLIP (para YOLOE) y Roboflow
!nvidia-smi -L || echo "Active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip -q install -U ultralytics roboflow opencv-python-headless pandas matplotlib
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

In [ ]:
# 2. Definición de los 5 Keypoints de Llanta y Extractor Geométrico (YOLOE Zero-shot)
import os, glob, random, shutil, json, cv2, numpy as np, torch, pandas as pd
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
KPT_COLORES = [(255, 255, 255), (255, 0, 0), (0, 200, 255), (0, 255, 0), (255, 0, 255)]
ESQUELETO = [(0, 1), (0, 2), (0, 3), (0, 4)]
FLIP_IDX = [0, 1, 2, 4, 3]
PROMPTS_VEHICULO = {'motorcycle': 'moto', 'car': 'auto', 'sedan': 'auto', 'bus': 'bus',
                    'minibus': 'minibus', 'van': 'minibus', 'truck': 'camion'}
PROMPTS_LLANTA = ['wheel', 'tire', 'car wheel']

def kps_de_mascara(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 8:
        return None
    pts = c.reshape(-1, 2).astype(float)
    if len(pts) >= 5:
        (cx, cy), _, _ = cv2.fitEllipse(c)
    else:
        cx, cy = pts.mean(0)
    contacto = pts[pts[:, 1].argmax()]
    superior = pts[pts[:, 1].argmin()]
    izquierda = pts[pts[:, 0].argmin()]
    derecha = pts[pts[:, 0].argmax()]
    k = np.array([[cx, cy], contacto, superior, izquierda, derecha])
    return np.concatenate([k, np.ones((5, 1))], axis=1)

In [ ]:
# 3. Descarga del Dataset desde Roboflow (o Auto-etiquetado en formato YOLO-pose [5, 3])
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'TU_ROBOFLOW_API_KEY')
WORKSPACE = 'peaje-iam811'
PROYECTO_M3 = 'keypoints-llantas-ejes'
VERSION_M3 = 1

if ROBOFLOW_API_KEY != 'TU_ROBOFLOW_API_KEY':
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    dataset = rf.workspace(WORKSPACE).project(PROYECTO_M3).version(VERSION_M3).download('yolov11')
    DATA_YAML = os.path.join(dataset.location, 'data.yaml')
    print('Dataset M3 descargado de Roboflow:', DATA_YAML)
else:
    DATA_YAML = '/content/dataset_llantas/data.yaml'
    print('Para descargar de Roboflow configure ROBOFLOW_API_KEY o use la sección 8.2 de auto-etiquetado con YOLOE.')

In [ ]:
# 4. Entrenamiento de YOLO11s-pose con kpt_shape [5, 3] y exportación de m3_ruedas_pose.pt
if os.path.exists(DATA_YAML):
    modelo_pose = YOLO('yolo11s-pose.pt')
    modelo_pose.train(
        data=DATA_YAML,
        epochs=80,
        imgsz=640,
        batch=16,
        device=0,
        project='/content/runs/pose',
        name='llantas',
        exist_ok=True
    )
    RUTA_YOLOPOSE = '/content/runs/pose/llantas/weights/best.pt'
    metricas = YOLO(RUTA_YOLOPOSE).val(data=DATA_YAML)
    print('mAP50 de keypoints (5 puntos por llanta):', round(float(metricas.pose.map50), 4))
    shutil.copy(RUTA_YOLOPOSE, 'm3_ruedas_pose.pt')
    print('Pesos exportados a: m3_ruedas_pose.pt')

In [ ]:
# 5. Algoritmo Oficial de Conteo de Ejes (asociar + analizar_ejes)
UMBRAL_CONTENCION = 0.7
UMBRAL_CERCANA = 0.65
SEPARACION_EJE = 0.6

def altura_visible(l):
    return max(1.0, l['kps'][1, 1] - l['kps'][2, 1])

def analizar_ejes(v, ls):
    if not ls:
        return None, '-', 'Sin llantas detectadas'
    hmax = max(altura_visible(l) for l in ls)
    cercanas = [l for l in ls if altura_visible(l) >= UMBRAL_CERCANA * hmax]
    xs = sorted(l['kps'][0, 0] for l in cercanas)
    diam = float(np.median([altura_visible(l) for l in cercanas]))
    ejes = 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * diam)
    redondez = np.median([(l['kps'][4, 0] - l['kps'][3, 0]) / altura_visible(l) for l in cercanas])
    vista = 'lateral' if redondez > 0.8 else '3/4'
    nota = ''
    if ejes < 2:
        ejes, nota = 2, 'Mínimo asumido (solo se detectó 1 eje)'
    return ejes, vista, nota